In [7]:
import pandas as pd

url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

rows, cols = df.shape
print(f"Rows: {rows}")
print(f"Columns: {cols}")

survived_count = df['Survived'].value_counts()
print(f"\nSurvived: {survived_count.get(1, 0)}")
print(f"Died: {survived_count.get(0, 0)}")

Rows: 891
Columns: 12

Survived: 342
Died: 549


In [8]:
# Women OR Children (Age < 16) who did NOT survive
filtered_passengers = df[
    ((df['Sex'] == 'female') | (df['Age'] < 16)) & 
    (df['Survived'] == 0)
]

# Number of such passengers
count = len(filtered_passengers)
print(f"Women OR Children (<16) who did NOT survive: {count}")

# Optional: Show these passengers
print("\nFirst few passengers:")
print(filtered_passengers[['PassengerId', 'Name', 'Sex', 'Age', 'Survived']].head())

Women OR Children (<16) who did NOT survive: 100

First few passengers:
    PassengerId                                               Name     Sex  \
7             8                     Palsson, Master. Gosta Leonard    male   
14           15               Vestrom, Miss. Hulda Amanda Adolfina  female   
16           17                               Rice, Master. Eugene    male   
18           19  Vander Planke, Mrs. Julius (Emelia Maria Vande...  female   
24           25                      Palsson, Miss. Torborg Danira  female   

     Age  Survived  
7    2.0         0  
14  14.0         0  
16   2.0         0  
18  31.0         0  
24   8.0         0  


In [10]:
# 1. တန်းစီးခ (Fare) ရဲ့ မီဒီယံကို အတန်းအလိုက် (Pclass) ရှာရန်
median_fare_by_class = df.groupby('Pclass')['Fare'].median()
print("တန်းစီးခ မီဒီယံ (အတန်းအလိုက်):")
print(median_fare_by_class)

# 2. တန်းစီးခ အကွာအဝေး (maximum - minimum) ရှာရန်
fare_range_by_class = df.groupby('Pclass')['Fare'].agg(['min', 'max'])
fare_range_by_class['range'] = fare_range_by_class['max'] - fare_range_by_class['min']
print("\nတန်းစီးခ အကွာအဝေး (အတန်းအလိုက်):")
print(fare_range_by_class)

# 3. ဘယ်အတန်းမှာ အကွာအဝေးအကြီးဆုံးလဲ?
max_range_class = fare_range_by_class['range'].idxmax()
max_range_value = fare_range_by_class['range'].max()
print(f"\nအကွာအဝေးအကြီးဆုံး အတန်း: Pclass {max_range_class}")
print(f"အကွာအဝေး: {max_range_value:.2f}")

တန်းစီးခ မီဒီယံ (အတန်းအလိုက်):
Pclass
1    60.2875
2    14.2500
3     8.0500
Name: Fare, dtype: float64

တန်းစီးခ အကွာအဝေး (အတန်းအလိုက်):
        min       max     range
Pclass                         
1       0.0  512.3292  512.3292
2       0.0   73.5000   73.5000
3       0.0   69.5500   69.5500

အကွာအဝေးအကြီးဆုံး အတန်း: Pclass 1
အကွာအဝေး: 512.33


In [12]:
# Passenger count, survival rate, average age by Pclass and Embarked
result = df.groupby(['Pclass', 'Embarked']).agg(
    passengers=('PassengerId', 'count'),
    survival_rate=('Survived', lambda x: x.mean() * 100),
    avg_age=('Age', 'mean')
).round(2).reset_index()

print(result)

   Pclass Embarked  passengers  survival_rate  avg_age
0       1        C          85          69.41    38.03
1       1        Q           2          50.00    38.50
2       1        S         127          58.27    38.15
3       2        C          17          52.94    22.77
4       2        Q           3          66.67    43.50
5       2        S         164          46.34    30.39
6       3        C          66          37.88    20.74
7       3        Q          72          37.50    25.94
8       3        S         353          18.98    25.70


In [13]:
# Extract title from Name
df['Title'] = df['Name'].str.extract(r', ([A-Za-z]+)\.', expand=False)

# Show title counts
title_counts = df['Title'].value_counts()
print(title_counts)

Title
Mr          517
Miss        182
Mrs         125
Master       40
Dr            7
Rev           6
Mlle          2
Major         2
Col           2
Don           1
Mme           1
Ms            1
Sir           1
Lady          1
Capt          1
Jonkheer      1
Name: count, dtype: int64


In [14]:
# Create FamilySize column
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Calculate survival rate by FamilySize
survival_by_family = df.groupby('FamilySize')['Survived'].mean()

# Find max and min survival rates
max_survival_size = survival_by_family.idxmax()
min_survival_size = survival_by_family.idxmin()

print(f"FamilySize {max_survival_size}: Highest survival rate {survival_by_family[max_survival_size]:.2%}")
print(f"FamilySize {min_survival_size}: Lowest survival rate {survival_by_family[min_survival_size]:.2%}")

FamilySize 4: Highest survival rate 72.41%
FamilySize 8: Lowest survival rate 0.00%


In [16]:
# Fill missing Age with median by Sex and Pclass
df['AgeFilled'] = df.groupby(['Sex', 'Pclass'])['Age'].transform(
    lambda x: x.fillna(x.median())
)

# Show some rows with previously missing Age
print(df[['Age', 'AgeFilled', 'Sex', 'Pclass']].head(10))

    Age  AgeFilled     Sex  Pclass
0  22.0       22.0    male       3
1  38.0       38.0  female       1
2  26.0       26.0  female       3
3  35.0       35.0  female       1
4  35.0       35.0    male       3
5   NaN       25.0    male       3
6  54.0       54.0    male       1
7   2.0        2.0    male       3
8  27.0       27.0  female       3
9  14.0       14.0  female       2


In [20]:
# Simple version
bins = [0, 12, 18, 60, 150]
labels = ['Child', 'Teen', 'Adult', 'Senior']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels, right=False)

result = df.groupby(['AgeGroup', 'Sex'])['Survived'].mean() * 100
print(result.unstack().round(1))

Sex       female  male
AgeGroup              
Child       59.4  55.6
Teen        82.6  13.6
Adult       76.7  18.0
Senior     100.0  13.6


C:\Users\Kaung Satt Lin\AppData\Local\Temp\ipykernel_18744\78873759.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  result = df.groupby(['AgeGroup', 'Sex'])['Survived'].mean() * 100


In [22]:
# Alternative URL 1 နဲ့စမ်းကြည့်ပါ
import pandas as pd
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'
df = pd.read_csv(url)
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [23]:
# Task 9: Deck analysis

# 1. Cabin (သို့) Deck column ကိုသုံးပါ
# မင်းဒေတာမှာ 'deck' column ရှိပြီးသား (NaN values ပါ)
print("1. Deck column ရှိပြီးသား:")
print(df['deck'].value_counts(dropna=False))

# 2. Deck သိရင် Survival rate
deck_survival = df[df['deck'].notna()].groupby('deck')['survived'].mean() * 100
print("\n2. Deck အလိုက် Survival %:")
print(deck_survival.round(1))

# 3. အစုံဆုံး Deck
if not deck_survival.empty:
    safest_deck = deck_survival.idxmax()
    print(f"\n3. အစုံဆုံး Deck: {safest_deck} ({deck_survival.max():.1f}%)")
else:
    print("\n3. Deck data မရှိပါ")

# 4. Deck နဲ့ Class ဆက်စပ်မှု
if 'deck' in df.columns:
    deck_class = pd.crosstab(df['deck'], df['pclass'], normalize='index') * 100
    print("\n4. Deck နဲ့ Pclass ဆက်စပ်မှု (%):")
    print(deck_class.round(1))

1. Deck column ရှိပြီးသား:
deck
NaN    688
C       59
B       47
D       33
E       32
A       15
F       13
G        4
Name: count, dtype: int64

2. Deck အလိုက် Survival %:
deck
A    46.7
B    74.5
C    59.3
D    75.8
E    75.0
F    61.5
G    50.0
Name: survived, dtype: float64

3. အစုံဆုံး Deck: D (75.8%)

4. Deck နဲ့ Pclass ဆက်စပ်မှု (%):
pclass      1     2      3
deck                      
A       100.0   0.0    0.0
B       100.0   0.0    0.0
C       100.0   0.0    0.0
D        87.9  12.1    0.0
E        78.1  12.5    9.4
F         0.0  61.5   38.5
G         0.0   0.0  100.0


In [26]:
# Task 10 အတွက် မှန်ကန်တဲ့ code
# 1. Priority သတ်မှတ်ရန်
df['Priority'] = 'Low'  # default
df.loc[((df['sex'] == 'female') | (df['age'] < 16)), 'Priority'] = 'High'

# 2. Each class အတွက် survival rates
result = df.groupby(['pclass', 'Priority'])['survived'].mean().unstack() * 100

# 3. Difference
result['Difference'] = result['High'] - result['Low']

print("Survival Rate by Class and Priority (%):")
print(result.round(1))

# 4. Best class for "Women and Children First"
best_class = result['Difference'].idxmax()
best_diff = result['Difference'].max()
print(f"\nBest class for 'Women & Children First': Class {best_class} ({best_diff:.1f}% difference)")

Survival Rate by Class and Priority (%):
Priority  High   Low  Difference
pclass                          
1         96.9  35.3        61.6
2         92.9   8.1        84.9
3         47.1  11.9        35.2

Best class for 'Women & Children First': Class 2 (84.9% difference)
